### **CPUC Data Request**: Localized air temperature and dewpoint temperature
- All available dynamically downscaled models in the Cal-Adapt: Analytics Engine catalog
- HadISD station data available in 'wecc-hadisd' bucket (private -- requires access)

Produces:
- netcdf file per station with air temperature and dewpoint temperature as separate variables
- summary stats csv file
    - Values represent the mulit-model mean count of dewpoint temperature exceeding air temperature per month
    
    
### Step 0: Import libraries

In [ ]:
import climakitae as ck
import xarray as xr
import pandas as pd
import hvplot as hv
import numpy as np
import matplotlib.pyplot as plt

from climakitae.util.utils import read_csv_file, get_closest_gridcell, convert_to_local_time

from xclim.core.calendar import convert_calendar
from xclim.core.units import convert_units_to
from xclim.sdba.adjustment import QuantileDeltaMapping
from xclim.sdba import Grouper

from bokeh.models import HoverTool
from timezonefinder import TimezoneFinder

import s3fs
import pyproj
import itertools

### Step 1: QDM function

In [ ]:
window = 90
def do_QDM(obs, ds, nquantiles=20, 
           group='time.dayofyear', window=window, 
           kind="+"):
    
    group = Grouper(group, window=window)

    ds.attrs['variable'] = ds.name
    ds.name = 'Raw' 
    
    QDM = QuantileDeltaMapping.train(
        obs, 
        ds.sel(
            time=slice(str(obs.time.values[0]),
                       str(obs.time.values[-1]))), 
        nquantiles=nquantiles, 
        group=group, 
        kind=kind)
    
    ds_adj = QDM.adjust(ds).compute()
    
    QDM_ds = QDM.ds.rename(dict(
        dayofyear = 'Day of Year', 
        quantiles='Quantile'))    
    
    ds_adj.name = 'Adjusted' 
    ds_adj = xr.merge([ds, ds_adj])
    
    return QDM_ds,ds_adj

In [ ]:
def get_closest_land_gridcell(data, lat, lon, res='3km', print_coords=True):
    """From input gridded data, get the closest gridcell to a lat, lon coordinate pair.

    This function first transforms the lat,lon coords to the gridded data’s projection.
    Then, it uses xarray’s built in method .sel to get the nearest gridcell.

    Parameters
    -----------
    data: xr.DataArray or xr.Dataset
        Gridded data
    lat: float
        Latitude of coordinate pair
    lon: float
        Longitude of coordinate pair
    res: str
        Spatial resolution for Santa Barbara station
    print_coords: bool, optional
        Print closest coorindates?
        Default to True. Set to False for backend use.

    Returns
    --------
    xr.DataArray
        Grid cell closest to input lat,lon coordinate pair

    See also
    --------
    xarray.DataArray.sel
    """
    # Make Transformer object
    lat_lon_to_model_projection = pyproj.Transformer.from_crs(
        crs_from="epsg:4326",  # Lat/lon
        crs_to=data.rio.crs,  # Model projection
        always_xy=True,
    )

    # Hard-coding for Santa Barbara, forces land pixel selection over water pixel
    # Note fine-tuning produces different results for different spatial resolutions, specify which on call
    if lat==34.424 and lon==-119.842: # station coordinates for SB (HadISD_72392523190)
        print('Selecting closest land pixel for Santa Barbara...')
        if res == '9km':
            lat = 34.43
            lon = -119.83
        elif res == '3km':
            lat = 34.43
            lon = -119.845

    # Convert coordinates to x,y
    x, y = lat_lon_to_model_projection.transform(lon, lat)

    # Get closest gridcell
    closest_gridcell = data.sel(x=x, y=y, method="nearest")

    # Output information
    if print_coords:
        print(
            "Input coordinates: (%.4f, %.4f)" % (lat, lon)
            + "\nNearest grid cell coordinates: (%.4f, %.4f)"
            % (closest_gridcell.lat.values.item(), closest_gridcell.lon.values.item())
        )
    return closest_gridcell

### Step 2: Run full batch mode for all available stations
Warning: Each station takes approximately **25 minutes to run**. All 70 stations will take **29 hours of continuous run time**. 

In [ ]:
# read stations df
stations = "/home/jovyan/cae-notebooks/collaborative/DFU/wecc-station-data.csv"
stations_df = read_csv_file(stations)

# prep summary stats df
stats_list = []

# Initialize selections
selections = ck.Select()

def tas_dew_localize(stations_df):
    for my_station in stations_df['icao']:
    # for my_station in ['KFSD', 'KMCI', 'KSGF']:    # outside of wecc stations, run on 45km
    # for my_station in ['KSBA']:                    # SB station - water pixel check

        ## STEP 1 =================================================================================================
        # Grab obsevations data
        station_id = str(stations_df[stations_df['icao'] == my_station]['station id'].values[0]).replace('-', '')
        print('Running tas+dpts localization on: HadISD_{} ({})'.format(station_id, my_station))

        s3 = s3fs.S3FileSystem(anon=False, key=AWS_ACCESS_KEY_ID, secret=AWS_SECRET_ACCESS_KEY)
        aws_path = "s3://wecc-hadisd/02_tmp_tas_dpt/"
        filepath_zarr = aws_path + "HadISD_{}.zarr".format(station_id)
        print('Opening: {}'.format(filepath_zarr))
        store = s3fs.S3Map(root=filepath_zarr, s3=s3, check=False)
        ds = xr.open_zarr(store=store, consolidated=True)

        ## STEP 2 =================================================================================================
        # Subset and run per variable
        
        vars_to_run = ['tas'] #, 'dpts']
        for var in vars_to_run:
            print('\nStart processing on {}'.format(var))
            obs_ds = ds[var] # Subset for variable
            obs_ds = convert_units_to(obs_ds, "degF") # Convert units from K to degF
            obs_ds = obs_ds.chunk(dict(time=-1)).compute()

            # extract coordinates
            lat0 = obs_ds.latitude.values
            lon0 = obs_ds.longitude.values
            print('Obs lat-lon coords: ', lat0, lon0)

            # station timezone conversion
            # cannot use the convert_to_local_time function as we are not pulling bias-corrected model data for a station 
            # but modifying the existing raw station data itself
            # 1981 - 2014 is our baseline period
            print('Starting timezone conversion...')
            obs_ds_data = obs_ds.loc[(obs_ds.time.dt.year >= 1981) & (obs_ds.time.dt.year <= 2014)]
            obs_ds_data.time # to keep

            # need to retrieve 2015 to grab the last ~8 hours of "2014" from 2015 (in UTC) to do timezone conversion
            obs_ds_tzslice = obs_ds.loc[obs_ds.time.dt.year == 2015]

            # combine and convert
            obs_ds_total = xr.concat([obs_ds_data, obs_ds_tzslice], dim='time') # 1981-2015
            tf = TimezoneFinder()
            local_tz = tf.timezone_at(lng=float(lon0), lat=float(lat0))
            new_time = (pd.DatetimeIndex(obs_ds_total.time)
                        .tz_localize("UTC")
                        .tz_convert(local_tz)
                        .tz_localize(None)
                        .astype("datetime64[ns]"))
            obs_ds_total['time'] = new_time

            # subset by initial time
            start = obs_ds_data.time[0]
            end = obs_ds_data.time[-1]
            obs_ds_local = obs_ds_total.sel(time=slice(start, end))
            print('obs timezone correction complete!')

            # retrieve WRF data
            print('Retrieve WRF data...')
            selections.scenario_historical=['Historical Climate']
            selections.scenario_ssp=['SSP 3-7.0 -- Business as Usual']
            selections.append_historical = True
            selections.area_average = 'No'
            selections.time_slice = (1981, 2100)
            selections.resolution = '45 km' # can only use 9km, not 3km for outside of CA regions
            selections.timescale = 'hourly'
            if var == 'tas':
                selections.variable = 'Air Temperature at 2m'
            elif var == 'dpts':
                selections.variable = 'Dew point temperature'
            selections.units = 'degF'
            selections.area_subset = 'lat/lon'
            selections.cached_area = ['coordinate selection']
            selections.latitude= (lat0-.3, lat0+.3) # for 9km spacing
            selections.longitude= (lon0-.3, lon0+.3) # for 9km spacing
            wrf_ds = selections.retrieve()
            print('Retrieving WRF lat-lon: ', selections.latitude, selections.longitude)
            # spacing on latlon kept larger because some station locs are tricky
            # keeping large spacing and reduce to single grid cell after tz correction

            # convert to local timezone for WRF data
            wrf_ds = convert_to_local_time(wrf_ds, selections)
            
            # reduce to only closest grid cell
            print('Retrieving closest LAND gridcell...')
            # wrf_ds = get_closest_land_gridcell(wrf_ds, lat0, lon0, res='3km', print_coords=True) ## because of sig figs, a slighlt different value is returned for non-KSBA stations
            wrf_ds = get_closest_gridcell(wrf_ds, lat0, lon0, print_coords=True)
            # need to unchunk for bias correction
            wrf_ds = wrf_ds.chunk(dict(time=-1)).compute()
            # do some renaming for plotting ease later
            wrf_ds.attrs['physical_variable'] = wrf_ds.name
            wrf_ds.name = 'Raw'

            # dropping duplicate time indexes and resetting calendar
            wrf_ds = wrf_ds.drop_duplicates(dim='time', keep='first')
            obs_ds = obs_ds_local.drop_duplicates(dim='time', keep='first')

            wrf_ds = convert_calendar(wrf_ds, "noleap")
            obs_ds = convert_calendar(obs_ds, "noleap")

            print('{} data now processed to go into localization process'.format(var))
            if var == 'tas':
                adj_factors1, adj_ds1 = do_QDM(obs_ds, wrf_ds)                
                # drop raw data variable (and unnecessary coordinates), and rename adjusted back to air temperature
                adj_ds1 = adj_ds1['Adjusted']
                adj_ds1.name = 'Adjusted Air Temperature at 2m'
                adj_ds1 = adj_ds1.squeeze()
                adj_ds1 = adj_ds1.reset_coords(names=['Lambert_Conformal','x','y','lakemask','landmask','lat','lon'], drop='True')
                print('QDM complete on {}'.format(var))

            elif var == 'dpts':
                adj_factors2, adj_ds2 = do_QDM(obs_ds, wrf_ds)
                # drop raw data variable (and unnecessary coordinates?), and rename adjusted back to air temperature
                adj_ds2 = adj_ds2['Adjusted']
                adj_ds2.name = 'Adjusted Dewpoint Temperature'
                adj_ds2 = adj_ds2.squeeze()
                adj_ds2 = adj_ds2.reset_coords(names=['Lambert_Conformal','x','y','lakemask','landmask','lat','lon'], drop='True')
                print('QDM complete on {}'.format(var))

        ## STEP 3 =================================================================================================
        # Merge arrays into one dataset
        merged_ds = xr.merge([adj_ds1, adj_ds2], compat='override')
        merged_ds = merged_ds.sel(time=slice('1981-01-01', '2100-12-31')) # ensuring right length
        print('\nds created')
        
        ## STEP 4 =================================================================================================
        # summary stats
        td_exceed_tas = merged_ds['Adjusted Dewpoint Temperature'] > merged_ds['Adjusted Air Temperature at 2m'] # number of instances where Td > T
        counts_per_month_sim = td_exceed_tas.groupby('time.month').sum().mean(dim='simulation')
        counts_per_month_sim_percent = (counts_per_month_sim.values / len(merged_ds['time'])) *100
        stats_list.append(counts_per_month_sim_percent)
        
        ## STEP 5 =================================================================================================
        # export data
        filename = 'bc_tas_dpts_HadISD_{}.nc'.format(station_id)
        ck.export(merged_ds, filename, 'NetCDF')

        #=====================================================================================
        # STEP 6
        # CLOSE DATASET TO SAVE MEMORY
        wrf_ds.close()
        obs_ds.close()
        merged_ds.close()
        print('All files closed \n')
        
    stats_df = pd.DataFrame(stats_list)    
    return stats_df
        
## Approximately takes 25 minutes per station
## total time to run for 70+ stations: ~30 hours
stats_df = tas_dew_localize(stations_df)

In [ ]:
# process and export summary stats dataframe
stats_df.to_csv('dewpt_exceed_tas_counts_station_localization.csv')

In [ ]:
# visualize 
sim = 'WRF_MIROC6_r1i1p1f1' # select a simulation to inspect

# Full time series
tas_plot = merged_ds['Adjusted Air Temperature at 2m'].sel(simulation=sim).plot(label='tas')
tdps_plot = merged_ds['Adjusted Dewpoint Temperature'].sel(simulation=sim).plot(label='dpts')

## Subset plots
# tas_plot = merged_ds['Adjusted Air Temperature at 2m'].sel(simulation=sim).sel(time=slice("2021-06-01", "2021-06-30")).plot(label='tas')
# tdps_plot = merged_ds['Adjusted Dewpoint Temperature'].sel(simulation=sim).sel(time=slice("2021-06-01", "2021-06-30")).plot(label='dpts')

plt.ylabel('Adjusted Temperatures [°F]');
plt.title('{}, sim {}'.format('KABQ', sim)); # change to which station is being visualized
plt.legend();